# SQL-аналіз витрат monobank

Ті самі дані, що й в `analysis.ipynb`, але відповіді на питання отримую **SQL-запитами** до бази SQLite.

**Що тут є з SQL:** `SELECT`, `WHERE`, `GROUP BY`, `ORDER BY`, `JOIN`, `CASE`, `CTE (WITH)`, підзапити, віконні функції (`SUM() OVER`, `ROW_NUMBER() OVER`).

> 🔒 Запити повертають лише кількості, частки (%) та індекси — без реальних сум. Файл бази `mono.db` у `.gitignore`.

## 1. Створюю базу й завантажую дані

In [1]:
import sqlite3
import pandas as pd

con = sqlite3.connect("mono.db")

def sql(query):
    """Виконати SQL-запит і повернути результат як таблицю pandas."""
    return pd.read_sql_query(query, con)

# таблиця транзакцій
tx = pd.read_csv("transactions.csv")
tx.to_sql("transactions", con, if_exists="replace", index=False)

# довідник категорій: MCC-код → категорія
MCC_GROUPS = {
    "Продукти":        [5411, 5412, 5422, 5441, 5451, 5462, 5499, 5331],
    "Кафе й ресторани": [5811, 5812, 5813, 5814],
    "Транспорт":       [4111, 4112, 4121, 4131, 4784, 4789, 7523],
    "Пальне":          [5541, 5542, 5983],
    "Аптеки й здоров'я": [5122, 5912, 8011, 8021, 8031, 8041, 8042, 8043, 8049, 8050, 8062, 8071, 8099],
    "Одяг і взуття":   [5611, 5621, 5631, 5641, 5651, 5655, 5661, 5691, 5699, 5948],
    "Краса":           [5977, 7230, 7297, 7298],
    "Електроніка":     [5045, 5732, 5734, 5735, 5946],
    "Підписки й digital": [4899, 5815, 5816, 5817, 5818, 7372],
    "Зв'язок":         [4812, 4814, 4816],
    "Розваги":         [7832, 7841, 7922, 7929, 7932, 7933, 7991, 7994, 7996, 7997, 7999, 5733],
    "Дім":             [5200, 5211, 5231, 5251, 5261, 5712, 5719, 5722],
    "Тварини":         [742, 5995],
    "Маркетплейси":    [5262],
    "Комунальні":      [4900],
    "Подарунки":       [5947],
    "Послуги":         [8999, 7299],
    "Готівка":         [6010, 6011],
    "Перекази":        [4829, 6012, 6051, 6536, 6537, 6538, 6540],
}
mcc_table = pd.DataFrame([(code, cat) for cat, codes in MCC_GROUPS.items() for code in codes],
                         columns=["mcc", "category"])
mcc_table.to_sql("mcc_categories", con, if_exists="replace", index=False)

sql("SELECT name FROM sqlite_master WHERE type = 'table'")

,name
0,transactions
1,mcc_categories


## 2. Подання (VIEW) «покупки»

`VIEW` — збережений запит, до якого далі звертаюся як до таблиці.
- `LEFT JOIN` приєднує категорію за MCC-кодом; якщо коду немає в довіднику — `COALESCE` підставляє «Інше»;
- `WHERE` залишає лише витрати (сума < 0) і прибирає перекази;
- `strftime` дістає з дати місяць, день тижня й годину.

In [2]:
con.execute("DROP VIEW IF EXISTS purchases")
con.execute("""
CREATE VIEW purchases AS
SELECT
    t.datetime,
    t.description,
    COALESCE(c.category, 'Інше')            AS category,
    -t.amount                                AS spent,
    strftime('%Y-%m', t.datetime)            AS month,
    CAST(strftime('%w', t.datetime) AS INT)  AS weekday,   -- 0 = неділя
    CAST(strftime('%H', t.datetime) AS INT)  AS hour
FROM transactions AS t
LEFT JOIN mcc_categories AS c ON c.mcc = t.mcc
WHERE t.amount < 0
  AND COALESCE(c.category, 'Інше') <> 'Перекази'
""")

sql("SELECT COUNT(*) AS покупок, COUNT(DISTINCT category) AS категорій, MIN(month) AS з, MAX(month) AS по FROM purchases")

,покупок,категорій,з,по
0,122,13,2026-05,2026-11


## 3. Структура витрат за категоріями

`GROUP BY` групує покупки за категорією, а **віконна функція** `SUM(...) OVER ()` рахує загальну суму по всіх рядках — так у тому ж запиті отримую частку кожної категорії у відсотках.

In [3]:
sql("""
SELECT
    category                                                   AS категорія,
    COUNT(*)                                                   AS покупок,
    ROUND(100.0 * SUM(spent) / SUM(SUM(spent)) OVER (), 1)     AS частка_витрат_pct
FROM purchases
GROUP BY category
ORDER BY частка_витрат_pct DESC
""")

,категорія,покупок,частка_витрат_pct
0,Краса,24,31.7
1,Продукти,35,21.8
2,Одяг і взуття,15,11.5
3,Кафе й ресторани,8,8.9
4,Маркетплейси,4,4.7
5,Транспорт,4,4.0
6,Комунальні,5,3.9
7,Подарунки,1,3.8
8,Підписки й digital,7,3.0
9,Зв'язок,11,2.8


## 4. Динаміка по місяцях (індекс: середній місяць = 100)

**CTE** (`WITH monthly AS (...)`) — тимчасова іменована таблиця всередині запиту: спочатку рахую суму за кожен місяць, потім ділю на середнє через `AVG(...) OVER ()`. **Підзапити** `(SELECT MIN(month) ...)` прибирають неповні перший і останній місяці.

In [4]:
sql("""
WITH monthly AS (
    SELECT month, SUM(spent) AS total, COUNT(*) AS n
    FROM purchases
    -- перший і останній місяці виписки неповні — прибираю їх підзапитами
    WHERE month > (SELECT MIN(month) FROM purchases)
      AND month < (SELECT MAX(month) FROM purchases)
    GROUP BY month
)
SELECT
    month                                              AS місяць,
    n                                                  AS покупок,
    ROUND(100.0 * total / AVG(total) OVER (), 1)       AS індекс
FROM monthly
ORDER BY month
""")

,місяць,покупок,індекс
0,2026-06,62,260.9
1,2026-07,33,197.3
2,2026-08,9,32.3
3,2026-09,6,3.3
4,2026-10,4,6.2


## 5. Будні чи вихідні, і в який час

`CASE WHEN` — умова всередині запиту: перетворюю номер дня в «будні/вихідні», а годину — в частину доби.

In [5]:
sql("""
SELECT
    CASE WHEN weekday IN (0, 6) THEN 'вихідні' ELSE 'будні' END   AS дні,
    CASE
        WHEN hour < 12 THEN '1) ранок (до 12)'
        WHEN hour < 17 THEN '2) день (12–17)'
        ELSE                '3) вечір (після 17)'
    END                                                           AS частина_доби,
    COUNT(*)                                                      AS покупок,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1)            AS частка_покупок_pct
FROM purchases
GROUP BY дні, частина_доби
ORDER BY дні, частина_доби
""")

,дні,частина_доби,покупок,частка_покупок_pct
0,будні,1) ранок (до 12),24,19.7
1,будні,2) день (12–17),37,30.3
2,будні,3) вечір (після 17),25,20.5
3,вихідні,1) ранок (до 12),10,8.2
4,вихідні,2) день (12–17),13,10.7
5,вихідні,3) вечір (після 17),13,10.7


## 6. Топ-3 категорії в кожному місяці

Віконна функція `ROW_NUMBER() OVER (PARTITION BY month ORDER BY ...)` нумерує категорії **окремо всередині кожного місяця** від найбільшої до найменшої — потім залишаю перші три.

In [6]:
sql("""
WITH by_month AS (
    SELECT
        month,
        category,
        SUM(spent) AS total,
        ROW_NUMBER() OVER (PARTITION BY month ORDER BY SUM(spent) DESC)  AS rnk,
        SUM(SUM(spent)) OVER (PARTITION BY month)                        AS month_total
    FROM purchases
    GROUP BY month, category
)
SELECT
    month                                     AS місяць,
    rnk                                       AS місце,
    category                                  AS категорія,
    ROUND(100.0 * total / month_total, 1)     AS частка_в_місяці_pct
FROM by_month
WHERE rnk <= 3
ORDER BY month, rnk
""")

,місяць,місце,категорія,частка_в_місяці_pct
0,2026-05,1,Краса,38.3
1,2026-05,2,Продукти,36.2
2,2026-05,3,Кафе й ресторани,25.5
3,2026-06,1,Продукти,29.1
4,2026-06,2,Одяг і взуття,16.3
5,2026-06,3,Краса,15.1
6,2026-07,1,Краса,61.0
7,2026-07,2,Продукти,14.8
8,2026-07,3,Кафе й ресторани,9.8
9,2026-08,1,Комунальні,42.5


## 7. Наскільки витрати сконцентровані

Скільки категорій дають 80% усіх витрат? Накопичувальна сума `SUM(...) OVER (ORDER BY ...)` додає частки по черзі від найбільшої категорії.

In [7]:
sql("""
WITH shares AS (
    SELECT category, 100.0 * SUM(spent) / SUM(SUM(spent)) OVER () AS pct
    FROM purchases
    GROUP BY category
)
SELECT
    ROW_NUMBER() OVER (ORDER BY pct DESC)                AS "#",
    category                                             AS категорія,
    ROUND(pct, 1)                                        AS частка_pct,
    ROUND(SUM(pct) OVER (ORDER BY pct DESC), 1)          AS накопичено_pct
FROM shares
ORDER BY pct DESC
""")

,#,категорія,частка_pct,накопичено_pct
0,1,Краса,31.7,31.7
1,2,Продукти,21.8,53.5
2,3,Одяг і взуття,11.5,65.0
3,4,Кафе й ресторани,8.9,73.9
4,5,Маркетплейси,4.7,78.5
5,6,Транспорт,4.0,82.6
6,7,Комунальні,3.9,86.5
7,8,Подарунки,3.8,90.2
8,9,Підписки й digital,3.0,93.3
9,10,Зв'язок,2.8,96.1


## 8. Перевірка: SQL vs pandas

Рахую частки категорій двома способами й перевіряю, що результати збігаються — так переконуюся, що запити правильні.

In [8]:
sql_result = sql("""
SELECT category, 100.0 * SUM(spent) / SUM(SUM(spent)) OVER () AS pct
FROM purchases GROUP BY category
""").set_index("category")["pct"].sort_index()

mcc_to_cat = dict(zip(mcc_table.mcc, mcc_table.category))
p = tx.assign(category=tx.mcc.map(mcc_to_cat).fillna("Інше"))
p = p[(p.amount < 0) & (p.category != "Перекази")]
pandas_result = (p.groupby("category").amount.sum() / p.amount.sum() * 100).sort_index()

diff = (sql_result - pandas_result).abs().max()
print(f"Максимальна розбіжність: {diff:.10f} п.п.")
print("✅ SQL і pandas дають однаковий результат" if diff < 1e-9 else "❌ Є розбіжності — треба шукати помилку")

Максимальна розбіжність: 0.0000000000 п.п.
✅ SQL і pandas дають однаковий результат


## Висновки з SQL-частини

1. **80% витрат дають лише 6 категорій із 13** (краса, продукти, одяг, кафе, маркетплейси, транспорт) — витрати сильно сконцентровані.
2. **70% покупок — у будні**, найбільше — **вдень з 12 до 17 (30%)**; у вихідні покупки розподілені по дню рівномірно.
3. **Структура змінилася з серпня:** у травні–липні лідирували краса й продукти (у липні краса — 61% витрат місяця), а з серпня на перших місцях лише регулярні платежі — комунальні, зв'язок, підписки, послуги. Після переїзду до Польщі українська картка залишилася для обов'язкових платежів в Україні — це підтверджує висновок про падіння витрат з `analysis.ipynb`.
4. Результати SQL-запитів збіглися з pandas — розрахунки перевірено двома способами.

*Травень і листопад у виписці неповні, тож їхні дані менш показові.*

In [9]:
con.close()